# Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells Exploration with `mlcroissant`
This notebook provides a template for loading and exploring the FAIR^2 dataset using the `mlcroissant` library.

### Dataset Source
The dataset source is provided via a Croissant schema URL.

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd
import json
import warnings
warnings.filterwarnings("ignore")  # suppress possible pandas warnings in display

# Define the dataset URL
url = 'https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json'

# Load the dataset metadata
dataset = mlc.Dataset(url)
metadata = dataset.metadata.to_json()
print(f"Dataset Name: {dataset.metadata.name}")
print(f"Description: {dataset.metadata.description}")
print(f"Croissant Schema URL: {url}")

## 2. Data Overview
Review available record sets, fields, and their IDs.

The FAIR² dataset may contain several record sets, each defined by a unique `@id`. We will list the available record sets in the dataset, and for each record set, display the available fields (columns) and their respective `@id`s.

In [ ]:
# List all available record sets and their fields using their '@id' as reference
record_set_ids = [rs['@id'] for rs in dataset.metadata.to_json().get('recordSet', [])]
if not record_set_ids:
    print('No record sets found in the dataset metadata. Inspecting distributions for data files...')
# Fallback: List distributions if no explicit record sets
distribution_list = dataset.metadata.to_json().get('distribution', [])
distribution_ids = [dist['@id'] for dist in distribution_list]
for idx, dist in enumerate(distribution_list):
    print(f"Distribution {idx}: @id={dist['@id']}, format={dist.get('encodingFormat','?')}")

# If possible, show record sets and fields
for rs in dataset.metadata.to_json().get('recordSet', []):
    print(f"RecordSet: @id={rs['@id']}, name={rs.get('name','')}")
    fields = rs.get('field', [])
    if isinstance(fields, dict):
        fields = [fields]
    elif isinstance(fields, list):
        fields = fields
    else:
        fields = []
    for fld in fields:
        if isinstance(fld, dict):
            print(f"    Field: @id={fld['@id']}, name={fld.get('name','')}, dataType={fld.get('dataType','')}")

## 3. Data Extraction
Load data from a specific record set into a DataFrame for analysis. Use the `@id` values listed above.

If the dataset does not provide explicit `recordSet` entries, you can load data from the primary distribution instead.

In [ ]:
# To extract data: load from each available record set (or the main distribution if no record sets).
dataframes = {}

record_sets = [rs['@id'] for rs in dataset.metadata.to_json().get('recordSet', [])]
if not record_sets:
    print("[Info] No recordSet entities defined. Loading from available distribution if possible...")
    # Some Croissant datasets only provide distributions. Try to extract with a None record_set.
    try:
        records = list(dataset.records(record_set=None))
        df = pd.DataFrame(records)
        dataframes[None] = df
        print(f"Loaded data available columns: {list(df.columns)}")
        display(df.head())
    except Exception as e:
        print(f"Could not load data: {e}")
else:
    for record_set_id in record_sets:
        try:
            records = list(dataset.records(record_set=record_set_id))
            df = pd.DataFrame(records)
            dataframes[record_set_id] = df
            print(f"Loaded {len(df)} records from RecordSet @id={record_set_id}")
            print(f"Columns: {list(df.columns)}")
            display(df.head())
        except Exception as e:
            print(f"Could not load records for RecordSet {record_set_id}: {e}")

## 4. Exploratory Data Analysis (EDA)
Apply common data processing steps, such as filtering records based on specific criteria, normalizing numeric fields, and categorizing data.
This section includes:
- Filtering outliers
- Normalizing values
- Grouping by attributes

**All field references use the field's `@id` as the column name.**

You can find and choose a numeric field from the loaded DataFrame columns above.

In [ ]:
# Pick a DataFrame from the loaded data
df = None
active_record_set_id = None
if dataframes:
    active_record_set_id = list(dataframes.keys())[0]
    df = dataframes[active_record_set_id]
else:
    print("No DataFrames loaded.")

# Identify numeric columns
numeric_cols = []
if df is not None:
    numeric_cols = df.select_dtypes(include=['number']).columns.tolist()

if numeric_cols:
    print(f"Numeric columns detected (likely field @id): {numeric_cols}")
    numeric_field = numeric_cols[0]
    threshold = df[numeric_field].mean()  # example threshold: filter above the mean
    filtered_df = df[df[numeric_field] > threshold].copy()
    print(f"Filtered records with {numeric_field} > {threshold:.2f} (field @id)")
    display(filtered_df.head())

    # Normalize the selected numeric field
    norm_colname = f"{numeric_field}_normalized"
    filtered_df[norm_colname] = (filtered_df[numeric_field] - filtered_df[numeric_field].mean()) / filtered_df[numeric_field].std()
    print(f"Normalized numeric field ({numeric_field}) added as {norm_colname}:")
    display(filtered_df[[numeric_field, norm_colname]].head())

    # Attempt to group by a categorical column if available
    cat_cols = df.select_dtypes(include=['object']).columns.tolist()
    group_field = None
    for c in cat_cols:
        # Pick the first column with few unique values
        if df[c].nunique() > 1 and df[c].nunique() < 20:
            group_field = c
            break

    if group_field:
        grouped_df = filtered_df.groupby(group_field)[numeric_field].mean().to_frame()
        print(f"Grouped by {group_field} (field @id):")
        display(grouped_df.head())
else:
    print("No numeric fields detected for EDA.")

## 5. Visualization
Visualize data distributions or relationships between fields in the dataset.

Below is a basic histogram and boxplot of the selected numeric field (by field `@id`).

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns
%matplotlib inline

if df is not None and numeric_cols:
    plt.figure(figsize=(10, 4))
    plt.subplot(1, 2, 1)
    sns.histplot(df[numeric_field].dropna(), bins=30)
    plt.title(f"Histogram of {numeric_field} (@id)")
    plt.xlabel(numeric_field)

    plt.subplot(1, 2, 2)
    sns.boxplot(x=df[numeric_field].dropna())
    plt.title(f"Boxplot of {numeric_field} (@id)")
    plt.xlabel(numeric_field)

    plt.tight_layout()
    plt.show()

    # If a group field was found, show a bar plot
    if group_field:
        plt.figure(figsize=(10,5))
        group_means = df.groupby(group_field)[numeric_field].mean().sort_values()
        sns.barplot(x=group_means.index, y=group_means.values)
        plt.title(f"Mean {numeric_field} by {group_field} (@id)")
        plt.xlabel(group_field)
        plt.ylabel(f"Mean {numeric_field}")
        plt.xticks(rotation=45)
        plt.tight_layout()
        plt.show()
else:
    print("No numeric field data available for visualization.")

## 6. Conclusion
Summarize key findings and observations from the dataset exploration.

- This FAIR^2 dataset provides mass spectrometry analysis results for proteins in extracellular vesicles from stimulated human mast cells.
- We demonstrated loading data, inspection of available fields by their `@id`, filtering, normalization, grouping, and visualizing record statistics using Python and `mlcroissant`.
- For advanced analysis, reference the dataset schema (via provided Croissant URL) and use field `@id` to ensure unambiguous feature selection and reproducibility.

**For more complex analysis, consult the [mlcroissant documentation](https://mlcommons.github.io/croissant/python/) and the complete Croissant schema for this dataset.**